# Chapter 5 — Methods and Methodologies
### Notebook 4 · Agentic lab — auditing, and planning under prerequisites

*Book reference: Extends §5.1–5.2*

An auditor agent that recommends a methodology and finds OntoClean violations, plus the course's first **planning MDP**: actions with prerequisites, rewarded by measured competency-question coverage.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch05_toolkit as ch5
from oe_course.sparql import SparqlStore
from oe_course.data import corpus
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
import ch05_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


**By the end of this notebook you can:**

1. Build an agent for a task where **a reasoner is useless** — the errors are ontological, not logical.
2. Plan a project as an MDP with **precedence constraints**, where skipping a step makes later steps unavailable.
3. Diagnose a **second-order** optimisation failure: a rule that only becomes visible after another rule is learned.
4. Recognise a reward model that rewards the wrong plan — and fix it.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

Note what is *not* here: a reasoner. Every taxonomy in this lab is consistent, so a tableau would return "fine" on all of them. The tools an agent needs here are meta-property lookups and constraint checks.

In [4]:
ctx = AG.Ch5Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:30s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":30s} {t.description.splitlines()[0]}')

list_methodologies             []
                               List the methodologies with the project signals each one fits.
recommend_methodology          ['brief']
                               Score every methodology against a project brief and rank them.
ontoclean_tags                 ['class_name']
                               Look up a class's OntoClean meta-properties (rigidity, identity, unity, dependence).
check_taxonomy                 ['axioms']
                               Check subsumption axioms ('Sub <= Super', one per line) against OntoClean.
list_constraints               []
                               Explain the OntoClean taxonomy constraints and why each one matters.
competency_question_coverage   []
                               Measure what fraction of the competency questions the ontology answers,
perform_step                   ['step']
                               Perform a development step, if its prerequisites are complete.


In [5]:
print(tools['ontoclean_tags'].invoke({'class_name': 'Student'}))
print(tools['ontoclean_tags'].invoke({'class_name': 'Person'}))
print()
print(tools['check_taxonomy'].invoke({'axioms': 'Person <= Student\nStudent <= Person'}))
print('\ntrajectory:', ctx.log.names())

{"class": "Student", "known": true, "rigidity": "~R", "identity": "-I", "unity": "-U", "dependence": "+D"}
{"class": "Person", "known": true, "rigidity": "+R", "identity": "+I", "unity": "+U", "dependence": "-D"}

[{"constraint": "anti-rigid-cannot-subsume-rigid", "axiom": "Person <= Student", "detail": "Student is anti-rigid (~R) but Person is rigid (+R)"}, {"constraint": "dependent-cannot-subsume-independent", "axiom": "Person <= Student", "detail": "Student is dependent (+D) but Person is independent (-D)"}]

trajectory: ['ontoclean_tags', 'ontoclean_tags', 'check_taxonomy']


### The tools also drive a project

`perform_step` refuses to run a step whose prerequisites are missing — the same precedence structure the MDP formalises below.

In [6]:
ctx2 = AG.Ch5Context()
t2 = {t.name: t for t in AG.build_toolset(ctx2)}
print('try axioms first :', t2['perform_step'].invoke({'step': 'axioms'}))
for step in ['requirements', 'competency_questions', 'taxonomy', 'axioms']:
    print(f'{step:22s}', t2['perform_step'].invoke({'step': step}))

try axioms first : {"error": "cannot do 'axioms' yet", "missing": ["taxonomy"]}


requirements           {"completed": ["requirements"], "coverage": 0.0}
competency_questions   {"completed": ["competency_questions", "requirements"], "coverage": 0.0}
taxonomy               {"completed": ["competency_questions", "requirements", "taxonomy"], "coverage": 0.333}
axioms                 {"completed": ["axioms", "competency_questions", "requirements", "taxonomy"], "coverage": 0.833}


## 2. The dataset

Ten cases, each a brief plus a taxonomy. The two halves are independent so the metric can say *which* half an agent is failing. The split is stratified so both halves cover all five methodologies and both clean and violating taxonomies.

In [7]:
all_cases = AG.build_dataset('all')
print(pd.DataFrame([{'id': e.id, 'methodology': e.gold_methodology,
                     'violations': len(e.gold_violations)} for e in all_cases]
                   ).to_string(index=False))
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print('\ntrain methodologies:', sorted({e.gold_methodology for e in train}))
print('dev   methodologies:', sorted({e.gold_methodology for e in dev}))

                 id     methodology  violations
      reuse-project            neon           0
     greenfield-lab    methontology           1
         consortium        diligent           1
      agile-startup           samod           0
  feasibility-first on-to-knowledge           1
    reuse-databases            neon           2
   clean-greenfield    methontology           0
 evolving-community        diligent           2
        sprint-team           samod           1
pilot-business-case on-to-knowledge           0

train methodologies: ['diligent', 'methontology', 'neon', 'on-to-knowledge', 'samod']
dev   methodologies: ['diligent', 'methontology', 'neon', 'on-to-knowledge', 'samod']


In [8]:
example = dev[0]
print('brief   :', example.brief)
print('taxonomy:')
print('  ' + example.taxonomy.replace('\n', '\n  '))
print('gold    :', example.gold_methodology, '|', example.gold_violations)

brief   : Greenfield build from scratch by a single team, covering the full lifecycle.
taxonomy:
  Person <= Student
  Person <= Entity
gold    : methontology | ['Person <= Student']


## 3. Baseline and GEPA

The un-instructed agent does what an inexperienced engineer does: names METHONTOLOGY because it is the one everyone has heard of, and reports no violations because the axioms all look fine — which, logically, they are.

In [9]:
lm = llm.configure_dspy(AG.AUDIT_RULEBOOK, AG.audit_responder)
baseline = AG.AuditProgram()
pred = baseline(**example.inputs())
print('recommended:', pred.methodology, '| violations:', pred.violations)
report = AG.audit_scorer(example, pred)
print('score      :', report.score)
for n in report.notes:
    print('   ', n)

recommended: methontology | violations: []
score      : 0.5
    Missed OntoClean violations: ['Person <= Student'].
    methodology_ok=1 violation_f1=0.00


In [10]:
before = ev.evaluate_dataset(baseline, dev, AG.audit_scorer)
print('BEFORE:', before['mean_score'])
print('violations:', before['violations'])

BEFORE: 0.3
violations: {'apply-ontoclean-constraints': 3, 'choose-samod': 1, 'choose-neon': 1, 'choose-diligent': 1, 'choose-on-to-knowledge': 1}


In [11]:
gepa_metric = ev.make_gepa_metric(AG.audit_scorer, AG.AUDIT_RULEBOOK)
reflect = llm.reflection_lm(AG.AUDIT_RULEBOOK, AG.audit_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=90, reflection_lm=reflect)
result = opt.compare(AG.AuditProgram(), tuned, dev, AG.audit_scorer)
print(result.report())

2026/08/17 07:39:54 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 90 metric calls of the program. This amounts to 9.00 full evals on the train+val set.


2026/08/17 07:39:54 INFO dspy.teleprompt.gepa.gepa: Using 5 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


GEPA Optimization:   0%|          | 0/90 [00:00<?, ?rollouts/s]

2026/08/17 07:39:54 INFO dspy.evaluate.evaluate: Average Metric: 1.5 / 5 (30.0%)


2026/08/17 07:39:54 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.3


...

2026/08/17 07:39:56 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/17 07:39:56 INFO dspy.teleprompt.gepa.gepa: Iteration 36: All subsample scores perfect. Skipping.


2026/08/17 07:39:56 INFO dspy.teleprompt.gepa.gepa: Iteration 36: Reflective mutation did not propose a new candidate


GEPA Optimization:  99%|█████████▉| 89/90 [00:01<00:00, 58.77rollouts/s]

mean score  0.300  ->  1.000   (delta +0.700)
violations  {'apply-ontoclean-constraints': 3, 'choose-samod': 1, 'choose-neon': 1, 'choose-diligent': 1, 'choose-on-to-knowledge': 1}
        ->  {}

instruction diff:
--- instruction (before)
+++ instruction (after)
@@ -1 +1,7 @@
 You are auditing an ontology project. Recommend a methodology and review the taxonomy.
+- RULE choose-samod: When the brief asks for agile, iterative or test-driven development in small increments, choose SAMOD.
+- RULE apply-ontoclean-constraints: Check every subsumption against the OntoClean constraints: an anti-rigid class cannot subsume a rigid one, an anti-unity class cannot subsume one with unity, and a dependent class cannot subsume an independent one.
+- RULE choose-on-to-knowledge: When the brief leads with a business case, feasibility or knowledge management pilot, choose On-To-Knowledge.
+- RULE choose-diligent: When the brief describes a distributed, decentralised or evolving effort with many contrib

## 4. A rule is learnable only if the data lets the agent break it

All six rules were discovered here. That is worth examining, because one of them very nearly could not have been.

`only-report-real-violations` punishes flagging a **sound** axiom as a violation. An agent can only commit that error if a sound axiom is present to be mis-flagged. Two of the training taxonomies deliberately mix a violating axiom with a sound one for exactly this reason — and if they did not, the rule would be unlearnable no matter how large the budget.

In [12]:
found = AG.AUDIT_RULEBOOK.active_in(result.instruction_after)
print('rules discovered:', sorted(found))
print('rules missed    :', sorted(set(AG.AUDIT_RULEBOOK.ids) - found) or 'none')
print()
for e in train:
    axioms = e.taxonomy.splitlines()
    sound = [a for a in axioms if a.strip() not in e.gold_violations]
    print(f'  {e.id:22s} {len(axioms)} axioms, {len(sound)} sound '
          f'-> over-reporting {"possible" if sound and e.gold_violations else "impossible"}')

rules discovered: ['apply-ontoclean-constraints', 'choose-diligent', 'choose-neon', 'choose-on-to-knowledge', 'choose-samod', 'only-report-real-violations']
rules missed    : none

  reuse-project          2 axioms, 2 sound -> over-reporting impossible
  consortium             2 axioms, 1 sound -> over-reporting possible
  feasibility-first      2 axioms, 1 sound -> over-reporting possible
  clean-greenfield       2 axioms, 2 sound -> over-reporting impossible
  sprint-team            1 axioms, 0 sound -> over-reporting impossible


> **The general principle**, which is easy to state and easy to forget:

> *A failure mode your evaluation data makes impossible is a failure mode your agent will keep in production.*

Exercise 4.1 removes the sound axioms and shows the rule disappearing.

## 5. Planning as an MDP with prerequisites

Every earlier MDP made all actions available at all times. This one does not:

| | |
|---|---|
| **S** | which development steps are done, and whether we shipped |
| **A** | perform a step **whose prerequisites are complete**, or ship |
| **T** | deterministic |
| **R** | −effort per step; on ship, the **measured** CQ coverage |

Precedence is the structural claim every methodology in §5.1 makes. Here it is enforced by the action set, and the reward comes from the coverage table you built in Notebook 1 — not from a stipulated number.

In [13]:
M = AG.MethodologyPlanMDP()
print(f'|S| = {len(M.states())}')
s0 = M.initial_state()
print('actions available at the start:', M.actions(s0))
print('\nNote what is NOT available: you cannot start with axioms or evaluation.')

|S| = 256
actions available at the start: ['requirements', 'ship']

Note what is NOT available: you cannot start with axioms or evaluation.


In [14]:
V, pi = mdp.value_iteration(M)
print(f'V*(s0) = {V[s0]:.3f}\n')
ep = mdp.run_episode(M, mdp.greedy_policy(pi))
for t in ep.transitions:
    print(f'  {str(t.state):10s} {t.action:22s} r={t.reward:+.2f}')
print(f'\noptimal plan: {" -> ".join(ep.actions)}')
print(f'return = {ep.discounted_return():.3f} '
      f'(coverage 1.0 minus {round(1.0 - ep.discounted_return(), 2)} of effort)')

V*(s0) = 0.250

  .......    requirements           r=-0.10
  R......    competency_questions   r=-0.10
  RC.....    taxonomy               r=-0.20
  RC.T...    axioms                 r=-0.25
  RC.TA..    instances              r=-0.10
  RC.TAI.    ship                   r=+1.00

optimal plan: requirements -> competency_questions -> taxonomy -> axioms -> instances -> ship
return = 0.250 (coverage 1.0 minus 0.75 of effort)


> **Now look at what the optimal plan skips.** It never does `reuse_search`, and — more uncomfortably — it never does `evaluation`. Both cost effort and neither unlocks a competency question, so under *this* reward they are pure loss.

That is not a bug in the solver. It is a **bug in the reward model**, and it is the same bug that makes real teams skip evaluation under deadline pressure: the measured objective does not credit it. Exercise 4.2 asks you to fix the reward rather than the plan.

In [15]:
skipped = [s for s in M.names if s not in ep.actions]
print('steps the optimal plan skips:', skipped)
for s in skipped:
    spec = ch5.DEVELOPMENT_STEPS[s]
    print(f'  {s:16s} cost={spec["cost"]:.2f} unlocks={spec["unlocks"] or "(nothing)"}')

steps the optimal plan skips: ['evaluation', 'reuse_search']
  evaluation       cost=0.10 unlocks=(nothing)
  reuse_search     cost=0.15 unlocks=(nothing)


### Exercise 4.1 — Make a rule unlearnable

Strip every **sound** axiom out of the training taxonomies, so over-reporting becomes impossible on that data. Re-run GEPA and show `only-report-real-violations` is no longer discovered — even though the dev set still punishes it.

> **Hint.** Keep only the axioms that appear in `gold_violations`.

In [16]:
# YOUR CODE HERE


<details>
<summary>Solution 4.1</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [17]:
import dspy
ablated = []
for e in train:
    keep = [a for a in e.taxonomy.splitlines() if a.strip() in e.gold_violations]
    ablated.append(dspy.Example(
        brief=e.brief, taxonomy='\n'.join(keep) or e.taxonomy,
        gold_methodology=e.gold_methodology, gold_violations=e.gold_violations,
        id=e.id).with_inputs('brief', 'taxonomy'))

print('ablated training taxonomies (violations only):')
for e in ablated:
    print(f'  {e.id:22s} {e.taxonomy.splitlines()}')

tuned_ablated = opt.run_gepa(AG.AuditProgram(), ablated, gepa_metric, valset=ablated,
                             max_metric_calls=90, reflection_lm=reflect)
res_ablated = opt.compare(AG.AuditProgram(), tuned_ablated, dev, AG.audit_scorer)
found_ablated = AG.AUDIT_RULEBOOK.active_in(res_ablated.instruction_after)

print('\nfull train    -> dev', result.after['mean_score'],
      '| rules', len(AG.AUDIT_RULEBOOK.active_in(result.instruction_after)))
print('ablated train -> dev', res_ablated.after['mean_score'],
      '| rules', len(found_ablated))
print('over-reporting rule learned?', 'only-report-real-violations' in found_ablated)
assert 'only-report-real-violations' not in found_ablated
print('\nThe rule vanished. Nothing about the agent, the metric or the budget\n'
      'changed -- only the data. Two axioms removed from a training set silently\n'
      'removed a capability, and the only symptom is a dev score 0.03 lower.\n'
      'In production the symptom would be an auditor that cries wolf.')

2026/08/17 07:40:05 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 90 metric calls of the program. This amounts to 9.00 full evals on the train+val set.


2026/08/17 07:40:05 INFO dspy.teleprompt.gepa.gepa: Using 5 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is just large enough to match your downstream task distribution, while providing as large trainset as possible.


ablated training taxonomies (violations only):
  reuse-project          ['Student <= Person', 'Person <= Entity']
  consortium             ['Statue <= Clay']
  feasibility-first      ['Person <= Pet']
  clean-greenfield       ['Student <= Person', 'Employee <= Person']
  sprint-team            ['Person <= Student']


GEPA Optimization:   0%|          | 0/90 [00:00<?, ?rollouts/s]

2026/08/17 07:40:05 INFO dspy.evaluate.evaluate: Average Metric: 1.5 / 5 (30.0%)


...

2026/08/17 07:40:07 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 2 (100.0%)


2026/08/17 07:40:07 INFO dspy.teleprompt.gepa.gepa: Iteration 36: All subsample scores perfect. Skipping.


2026/08/17 07:40:07 INFO dspy.teleprompt.gepa.gepa: Iteration 36: Reflective mutation did not propose a new candidate


GEPA Optimization:  99%|█████████▉| 89/90 [00:01<00:00, 45.50rollouts/s]


full train    -> dev 1.0 | rules 6
ablated train -> dev 0.9667 | rules 5
over-reporting rule learned? False

The rule vanished. Nothing about the agent, the metric or the budget
changed -- only the data. Two axioms removed from a training set silently
removed a capability, and the only symptom is a dev score 0.03 lower.
In production the symptom would be an auditor that cries wolf.


### Exercise 4.2 — Fix the reward so evaluation is worth doing

The optimal plan skips `evaluation`. Change the reward so that skipping it is penalised — for example, because unevaluated ontologies ship with defects — and show the optimal plan changing.

> **Hint.** Pass a custom `coverage_fn` to `MethodologyPlanMDP`.

In [18]:
# YOUR CODE HERE


<details>
<summary>Solution 4.2</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [19]:
def coverage_with_evaluation(done):
    """Shipping without evaluation loses a fifth of the delivered value.

    The justification is empirical, not moral: unevaluated ontologies ship with
    defects that cost more to fix later than the evaluation would have cost.
    """
    base = ch5.coverage_for_steps(done)
    return base if 'evaluation' in done else base * 0.8

M2 = AG.MethodologyPlanMDP(coverage_fn=coverage_with_evaluation)
V2, pi2 = mdp.value_iteration(M2)
ep2 = mdp.run_episode(M2, mdp.greedy_policy(pi2))
print('new optimal plan:', ' -> '.join(ep2.actions))
print(f"V* = {V2[M2.initial_state()]:.3f} (was {V[s0]:.3f})")
assert 'evaluation' in ep2.actions
print('\nEvaluation now pays for itself: 20%% of 1.0 coverage is 0.20, and the\n'
      'step costs 0.10. Nothing about the planner changed -- only what we told\n'
      'it to value. If your agents keep skipping something you care about, the\n'
      'first place to look is the reward, not the policy.')

new optimal plan: requirements -> competency_questions -> taxonomy -> axioms -> evaluation -> instances -> ship
V* = 0.150 (was 0.250)

Evaluation now pays for itself: 20%% of 1.0 coverage is 0.20, and the
step costs 0.10. Nothing about the planner changed -- only what we told
it to value. If your agents keep skipping something you care about, the
first place to look is the reward, not the policy.


### Exercise 4.3 — Make reuse worth searching for

`reuse_search` is also skipped. Model the NeOn claim — that reuse *reduces the cost* of building the taxonomy and axioms — and find the discount at which searching for reusable resources becomes optimal.

In [20]:
# YOUR CODE HERE


<details>
<summary>Solution 4.3</summary>

Run the cell below to check your answer against the reference implementation. The assertions are the grading criteria.

</details>

In [21]:
import copy
rows = []
for discount in [0.0, 0.2, 0.4, 0.6]:
    steps = copy.deepcopy(ch5.DEVELOPMENT_STEPS)
    # Modelling choice: reuse cannot make a step free, only cheaper.
    steps['taxonomy'] = dict(steps['taxonomy'],
                             requires=('competency_questions',))
    class ReuseMDP(AG.MethodologyPlanMDP):
        def transition(self, state, action):
            if action == 'ship':
                return [(1.0, type(state)(state.done, True), self.coverage(state.done))]
            cost = self.steps[action]['cost']
            if 'reuse_search' in state.done and action in ('taxonomy', 'axioms'):
                cost *= (1 - discount)
            return [(1.0, type(state)(state.done | {action}, False), -cost)]
    Mr = ReuseMDP(steps)
    Vr, pir = mdp.value_iteration(Mr)
    epr = mdp.run_episode(Mr, mdp.greedy_policy(pir))
    rows.append({'discount': discount, 'V*': round(Vr[Mr.initial_state()], 3),
                 'searches for reuse': 'reuse_search' in epr.actions})
print(pd.DataFrame(rows).to_string(index=False))
print('\nreuse_search costs 0.15 and can save at most discount x 0.45 (the cost of\n'
      'taxonomy + axioms), so it pays from roughly a third onwards. That is NeOn\n'
      'stated as an inequality: reuse is worth the search only when the resources\n'
      'you find genuinely displace work you would otherwise do.')

 discount   V*  searches for reuse
      0.0 0.25               False
      0.2 0.25               False
      0.4 0.28                True
      0.6 0.37                True

reuse_search costs 0.15 and can save at most discount x 0.45 (the cost of
taxonomy + axioms), so it pays from roughly a third onwards. That is NeOn
stated as an inequality: reuse is worth the search only when the resources
you find genuinely displace work you would otherwise do.


## Chapter 5 in the course arc

| | Ch. 1 | Ch. 2 | Ch. 3 | Ch. 4 | Ch. 5 |
|---|---|---|---|---|---|
| MDP | gather evidence | search a proof | budgeted, stochastic | construct | **plan under prerequisites** |
| grader | labels + judge | decision procedure | free oracle | labels + profiles | measured CQ coverage |
| what a reasoner buys you | nothing | — | everything | everything | **nothing** |

Chapter 5's contribution: the errors that matter most are often the ones your tooling cannot see. A reasoner amplifies whatever you assert — including your mistakes — so a methodology and a meta-property checker are not bureaucracy, they are the only defence against a whole class of bug.

Chapter 6 takes the repair that OntoClean could not express (`Statue <= Clay` is *constitution*, not subsumption) and gives you the vocabulary for it.